# Experiment 001 — Typo Robustness: Colab Pilot

Five cells: **setup → GPU install → fetch data → generate → report**.
Uses vLLM on a Colab T4 with Qwen2.5-1.5B-Instruct (ungated, no HF auth needed).
See `docs/PILOT_DECISIONS.md` for the two Regime B levers this pilot decides between.
See `docs/PROVENANCE.md` for version pinning rationale.

### Cell 1 — Setup: clone the repo and install the package

In [ ]:
# In Colab, clone your repository. Locally, skip the clone.
# !git clone https://github.com/natSegOS/glamor-research-onboarding.git
# %cd glamor-research-onboarding/experiments/001_typo_robustness

import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=False)
print("package installed")

### Cell 2 — Install the GPU stack

vLLM with CUDA support. Pinned for mutual compatibility; see docs/PROVENANCE.md §8.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-gpu.txt", "-q"], check=False)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### Cell 3 — Fetch the real Core-4 pilot subset

Pre-fetches 100 items per dataset from HuggingFace (no auth needed for these repos),
pinning the dataset revision SHA in `data/items/PROVENANCE.json`. Re-running this cell
overwrites cleanly with fresh SHAs — safe and idempotent.


In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "tools/build_task_items.py",
    "--reasoning-items", "100",   # pilot subset; scale to 600 for the main run
    "--mcq-items", "100",
    "--gsm-config", "main",
    "--seed", "1729",
    "--output-directory", "data/items",
], check=True)
print("items ready in data/items/")


### Cell 4 — Generate pilot results

Uses the pilot config (configs/pilot.yaml) and the Qwen2.5-1.5B-Instruct model, which
is ungated and fits on a T4. `is_confirmatory: false` in pilot.yaml means PIN_ME
revisions are allowed — no pre-registration step needed.

The runner is idempotent: kill and re-run at any point; it resumes from where it stopped.

In [ ]:
import subprocess
import sys

try:
    result = subprocess.run([
        sys.executable, "tools/run_generation.py",
        "--config", "configs/pilot.yaml",
        "--model", "qwen_1b5_pilot",
        "--output-directory", "results/pilot",
        "--git-commit", "unpinned",
    ], check=True, capture_output=True, text=True)
    
    print("generation done; see results/pilot/")

except subprocess.CalledProcessError as e:
    print(f"\n[ERROR] Script crashed!")
    print(f"Exit Code: {e.returncode}")
    print(f"--- Standard Error (stderr) --- \n{e.stderr}")
    print(f"--- Standard Output (stdout) --- \n{e.stdout}")

### Cell 5 — Build the results report and download it

In [ ]:
import subprocess, sys, pathlib

# Build the self-contained HTML report (global + local drill-down).
subprocess.run([
    sys.executable, "tools/build_report.py",
    "--generations", "results/pilot/pilot_generations.jsonl",
    "--output", "results/pilot/report.html",
], check=True)

# Download the report in Colab.
try:
    from google.colab import files
    files.download("results/pilot/report.html")
    print("report downloaded")
except ImportError:
    print("open results/pilot/report.html in your browser")
